In [2]:
# Machine Learning model training & evaluation

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# Step 1: Load the Preprocessed Dataset
# ==========================================
# Load the merged telecom dataset from the processed folder
telecom_customer_df = pd.read_csv('../data/processed/merged_telecom_data.csv')

print("Initial dataset shape:", telecom_customer_df.shape)

# ==========================================
# Step 2: Handle Missing Values (Null Imputation)
# ==========================================
# Fill numerical columns with median
numerical_columns = telecom_customer_df.select_dtypes(include=np.number).columns
for col in numerical_columns:
    if telecom_customer_df[col].isnull().sum() > 0:
        median_val = telecom_customer_df[col].median()
        telecom_customer_df[col] = telecom_customer_df[col].fillna(median_val)

# Fill categorical/text columns with 'Unknown'
categorical_columns = telecom_customer_df.select_dtypes(include=['object', 'str']).columns
for col in categorical_columns:
    if telecom_customer_df[col].isnull().sum() > 0:
        telecom_customer_df[col] = telecom_customer_df[col].fillna('Unknown')

# Verify that no null values remain
remaining_nulls = telecom_customer_df.isnull().sum().sum()
print(f"Remaining null values after cleaning: {remaining_nulls}")

# ==========================================
# Step 3: Feature Encoding (One-Hot Encoding)
# ==========================================
# Convert all categorical text columns into numerical binary dummy variables
telecom_customer_encoded_df = pd.get_dummies(telecom_customer_df, drop_first=True)

# Ensure all columns are strictly numeric (preventing any string conversion errors like 'U')
telecom_customer_encoded_df = telecom_customer_encoded_df.apply(pd.to_numeric, errors='coerce').fillna(0)

print(f"Dataset shape after One-Hot Encoding: {telecom_customer_encoded_df.shape}")

# ==========================================
# Step 4: Separate Features (X) and Target (y)
# ==========================================
# Drop 'Customer_ID' (unique identifier) and 'churn' (our target label) from features
X = telecom_customer_encoded_df.drop(columns=['Customer_ID', 'churn'], errors='ignore')
y = telecom_customer_encoded_df['churn']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")

# ==========================================
# Step 5: Train-Test Split (80% Train / 20% Test)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print("\n--- Train-Test Split Results ---")
print(f"Training features shape (X_train): {X_train.shape}")
print(f"Testing features shape (X_test): {X_test.shape}")
print(f"Training target shape (y_train): {y_train.shape}")
print(f"Testing target shape (y_test): {y_test.shape}")

# ==========================================
# Step 6: Train Random Forest Classifier Model
# ==========================================
print("\nTraining Random Forest Classifier model...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("Model training completed successfully!")

# ==========================================
# Step 7: Model Evaluation & Predictions
# ==========================================
# Predict on the test set
y_pred = rf_model.predict(X_test)

# Calculate accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

# Print detailed classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Initial dataset shape: (100000, 100)
Remaining null values after cleaning: 0
Dataset shape after One-Hot Encoding: (100000, 229)
Features shape (X): (100000, 227)
Target shape (y): (100000,)

--- Train-Test Split Results ---
Training features shape (X_train): (80000, 227)
Testing features shape (X_test): (20000, 227)
Training target shape (y_train): (80000,)
Testing target shape (y_test): (20000,)

Training Random Forest Classifier model...
Model training completed successfully!

Model Accuracy: 61.92%

Classification Report:

              precision    recall  f1-score   support

           0       0.62      0.63      0.62     10021
           1       0.62      0.61      0.62      9979

    accuracy                           0.62     20000
   macro avg       0.62      0.62      0.62     20000
weighted avg       0.62      0.62      0.62     20000



In [3]:
# ==========================================
# Step 8: Feature Importance Analysis
# ==========================================
# Extract feature importances from the trained Random Forest model
importances = rf_model.feature_importances_
feature_names = X.columns

# Create a DataFrame to sort top features
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort by highest importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("\n--- Top 10 Most Important Features for Churn ---")
print(feature_importance_df.head(10))


--- Top 10 Most Important Features for Churn ---
       Feature  Importance
27     eqpdays    0.030781
37  change_mou    0.023272
76      months    0.022751
29    mou_Mean    0.019452
4       totrev    0.019134
10      avgqty    0.019041
38  change_rev    0.019000
8       avgrev    0.018977
5       adjrev    0.018945
9       avgmou    0.018530


In [4]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ==========================================
# Step 9: Hyperparameter Tuning (Random Forest)
# ==========================================

# Define the hyperparameter grid to search through
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the base Random Forest model
base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Use RandomizedSearchCV for efficient optimization (cv=3 for faster processing on 80k rows)
rf_random_search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_grid,
    n_iter=5,          # Number of parameter combinations to try
    cv=3,              # 3-fold cross-validation
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Fit the RandomizedSearchCV to the training data
print("Starting Hyperparameter Tuning (this may take a short moment)...")
rf_random_search.fit(X_train, y_train)

# Print the best parameters found by the search
print("\nBest Hyperparameters Found:")
print(rf_random_search.best_params_)

# ==========================================
# Step 10: Evaluate the Tuned Model
# ==========================================
# Extract the best estimator model
best_rf_model = rf_random_search.best_estimator_

# Make predictions on the test set using the tuned model
y_pred_tuned = best_rf_model.predict(X_test)

# Calculate and print the new accuracy score
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"\nTuned Model Accuracy: {tuned_accuracy * 100:.2f}%")

# Print the detailed classification report for the tuned model
print("\nTuned Classification Report:\n")
print(classification_report(y_test, y_pred_tuned))

Starting Hyperparameter Tuning (this may take a short moment)...
Fitting 3 folds for each of 5 candidates, totalling 15 fits



Best Hyperparameters Found:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 30}

Tuned Model Accuracy: 63.24%

Tuned Classification Report:

              precision    recall  f1-score   support

           0       0.64      0.62      0.63     10021
           1       0.63      0.65      0.64      9979

    accuracy                           0.63     20000
   macro avg       0.63      0.63      0.63     20000
weighted avg       0.63      0.63      0.63     20000

